[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonasneves/colab-slm-playground/blob/main/trending_pytorch_chatbot_colab.ipynb)

# Trending PyTorch Chatbot

Fetches trending `text-generation` models from Hugging Face, lets you pick one, and runs a chat UI inside Colab.

**Requires a GPU runtime.** Go to Runtime → Change runtime type → T4 GPU (or better).

## 1 — Check GPU

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime → Change runtime type → T4 GPU, then re-run."
)

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU : {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
print()
print("Guidelines for this GPU:")
print("  fp16  → models up to ~7B params")
print("  4-bit → models up to ~13B params (lower quality, less memory)")

## 2 — Install dependencies

In [ ]:
!pip install -q transformers accelerate bitsandbytes huggingface_hub ipywidgets

## 3 — (Optional) Log in to Hugging Face

Required for gated models (Llama, Gemma, Mistral-instruct, etc.).
Skip this cell if you only plan to use open models.

In [ ]:
# Option A — token stored in Colab secrets (Colab Pro / paid)
# from google.colab import userdata
# from huggingface_hub import login
# login(token=userdata.get("HF_TOKEN"))

# Option B — paste token directly (never commit this)
# from huggingface_hub import login
# login(token="hf_...")

print("Skipped HF login. Uncomment one option above if you need gated models.")

## 4 — Fetch trending models

In [ ]:
from huggingface_hub import HfApi
from IPython.display import display, HTML

print("Fetching trending text-generation models...")

api = HfApi()
raw_models = list(api.list_models(
    pipeline_tag="text-generation",
    sort="trending_score",
    direction=-1,
    limit=40,
    full=True,
))

# Skip ONNX-only repos and estimate weight size from safetensors siblings
def _weight_size(siblings):
    if not siblings:
        return None
    weight_exts = (".safetensors", ".bin", ".pt")
    total = sum(
        s.size for s in siblings
        if any(s.rfilename.endswith(e) for e in weight_exts) and s.size
    )
    return total / 1e9 if total else None  # GB

model_info = []
for m in raw_models:
    # Skip repos that only have ONNX files
    libs = m.tags or []
    if "onnx" in libs and "pytorch" not in libs and "transformers" not in libs:
        continue
    size_gb = _weight_size(m.siblings)
    size_str = f"{size_gb:.1f} GB" if size_gb else "?"
    gated = getattr(m, "gated", False) or "gated" in libs
    model_info.append({
        "id": m.id,
        "downloads": m.downloads or 0,
        "likes": m.likes or 0,
        "size": size_str,
        "size_gb": size_gb,
        "gated": gated,
    })

rows = "".join(
    f"<tr>"
    f"<td style='padding:5px 10px'>{i+1}</td>"
    f"<td style='padding:5px 10px'><code>{m['id']}</code>{'&nbsp;🔒' if m['gated'] else ''}</td>"
    f"<td style='padding:5px 10px; text-align:right'>{m['downloads']:,}</td>"
    f"<td style='padding:5px 10px; text-align:right'>{m['likes']:,}</td>"
    f"<td style='padding:5px 10px; text-align:right'>{m['size']}</td>"
    f"</tr>"
    for i, m in enumerate(model_info)
)

display(HTML(f"""
<table style="border-collapse:collapse; font-family:monospace; font-size:13px">
  <thead>
    <tr style="background:#f0f0f0; border-bottom:2px solid #ccc">
      <th style="padding:6px 10px">#</th>
      <th style="padding:6px 10px; text-align:left">Model ID &nbsp; 🔒 = gated</th>
      <th style="padding:6px 10px">Downloads</th>
      <th style="padding:6px 10px">Likes</th>
      <th style="padding:6px 10px">Weight size</th>
    </tr>
  </thead>
  <tbody>{rows}</tbody>
</table>
"""))
print(f"Showing {len(model_info)} models.  🔒 = requires HF login.")

## 5 — Select model & precision

In [ ]:
import ipywidgets as widgets

model_dropdown = widgets.Dropdown(
    options=[(f"{m['id']}  ({m['size']})", m['id']) for m in model_info],
    description="Model:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="620px"),
)

precision_toggle = widgets.RadioButtons(
    options=[
        ("fp16  — full quality, needs ~2× model size in VRAM", "fp16"),
        ("4-bit — lower quality, roughly halves VRAM usage", "4bit"),
    ],
    description="Precision:",
    style={"description_width": "initial"},
)

display(model_dropdown, precision_toggle)
print("Pick a model and precision above, then run the next cell.")

---
### Notes

**Why does fp16 fit ~7B parameters on a 16 GB GPU?**
Each parameter stored in fp16 takes 2 bytes. A 7B-parameter model therefore needs ~14 GB just for weights, leaving ~2 GB for activations and the KV cache — tight but workable on a T4. fp32 would require ~28 GB, which exceeds the T4's VRAM entirely. `torch_dtype=torch.float16` with `device_map="auto"` handles this automatically.

**What is 4-bit quantization (NF4)?**
4-bit quantization stores each weight in 4 bits instead of 16, cutting memory roughly 4×. This notebook uses NF4 (NormalFloat4) with double quantization — the scheme introduced by [QLoRA](https://arxiv.org/abs/2305.14314). NF4 is designed specifically for normally-distributed neural network weights, so it preserves more information per bit than generic int4. The result: a 13B model fits in ~7 GB VRAM, at a modest quality cost versus fp16.

**What are gated models?**
Some model authors (Meta for Llama, Google for Gemma) require you to accept a license before downloading. Hugging Face enforces this via gated repos — your account token must be associated with an accepted license. Log in via cell 3, then the download proceeds normally. The 🔒 marker in the table flags these.

**`device_map="auto"`**
This tells `accelerate` to distribute the model across available devices automatically. On a single T4 it simply puts everything on the GPU. On multi-GPU setups or when the model is too large for VRAM alone, it splits layers across GPU and CPU RAM. It's the reason you don't need to manually call `.to("cuda")`.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID  = model_dropdown.value
PRECISION = precision_toggle.value

print(f"Loading : {MODEL_ID}")
print(f"Precision: {PRECISION}")
print("Downloading weights — this may take several minutes...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

load_kwargs = dict(
    device_map="auto",
    trust_remote_code=True,
)

if PRECISION == "4bit":
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
else:
    load_kwargs["torch_dtype"] = torch.float16

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Report actual VRAM used
allocated = torch.cuda.memory_allocated() / 1e9
print(f"\nReady. VRAM in use: {allocated:.1f} GB")

## 7 — Register callback & launch Chat UI

In [ ]:
import json
import torch
from google.colab import output as colab_output
from IPython.display import display, HTML
from IPython.display import JSON as IPJSON

MAX_NEW_TOKENS = 512


def _build_prompt(messages: list) -> str:
    if tokenizer.chat_template:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    parts = []
    for m in messages:
        role = "User" if m["role"] == "user" else "Assistant"
        parts.append(f"{role}: {m['content']}")
    return "\n".join(parts) + "\nAssistant:"


@torch.inference_mode()
def generate(messages: list) -> str:
    prompt  = _build_prompt(messages)
    inputs  = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    inputs  = {k: v.to(model.device) for k, v in inputs.items()}
    in_len  = inputs["input_ids"].shape[1]

    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.pad_token_id,
    )

    new_tokens = output_ids[0][in_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def _chat_callback(messages_json):
    try:
        messages = json.loads(messages_json)
        reply = generate(messages)
        return IPJSON({"reply": reply})
    except Exception as e:
        return IPJSON({"error": str(e)})


colab_output.register_callback("notebook.chat", _chat_callback)

short_name = MODEL_ID.split("/")[-1]
precision_label = "4-bit" if PRECISION == "4bit" else "fp16"

CHAT_HTML = f"""
<div id="hf-chat" style="
  font-family: 'Segoe UI', system-ui, -apple-system, sans-serif;
  max-width: 640px;
  border: 1px solid #d0d0d0;
  border-radius: 12px;
  overflow: hidden;
  background: #fafafa;
">
  <div style="
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
    color: white;
    padding: 14px 18px;
    font-size: 15px;
    font-weight: 600;
    display: flex;
    align-items: center;
    gap: 8px;
  ">
    <span style="font-size: 20px;">&#x1F916;</span>
    {short_name}
    &nbsp;<span style="font-weight:400; opacity:0.7; font-size:12px;">{precision_label} &middot; GPU</span>
  </div>

  <div id="hf-messages" style="
    height: 380px;
    overflow-y: auto;
    padding: 16px;
    display: flex;
    flex-direction: column;
    gap: 10px;
  "></div>

  <div style="
    display: flex;
    border-top: 1px solid #e0e0e0;
    background: white;
    padding: 8px;
    gap: 8px;
  ">
    <input id="hf-input" type="text" placeholder="Type a message..."
      style="
        flex: 1;
        border: 1px solid #d0d0d0;
        border-radius: 8px;
        padding: 10px 14px;
        font-size: 14px;
        outline: none;
      "
    />
    <button id="hf-send" onclick="hfSend()" style="
      background: #1a1a2e;
      color: white;
      border: none;
      border-radius: 8px;
      padding: 10px 18px;
      font-size: 14px;
      cursor: pointer;
      font-weight: 600;
    ">Send</button>
  </div>
</div>

<script>
(function() {{
  const messagesDiv = document.getElementById("hf-messages");
  const inputEl    = document.getElementById("hf-input");
  const sendBtn    = document.getElementById("hf-send");
  let history = [];

  function addBubble(role, text) {{
    const isUser = role === "user";
    const bubble = document.createElement("div");
    bubble.style.cssText = `
      max-width: 82%;
      padding: 10px 14px;
      border-radius: 12px;
      font-size: 14px;
      line-height: 1.5;
      word-wrap: break-word;
      white-space: pre-wrap;
      align-self: ${{isUser ? "flex-end" : "flex-start"}};
      background: ${{isUser ? "#1a1a2e" : "#e8e8e8"}};
      color: ${{isUser ? "#fff" : "#1a1a1a"}};
    `;
    bubble.textContent = text;
    messagesDiv.appendChild(bubble);
    messagesDiv.scrollTop = messagesDiv.scrollHeight;
    return bubble;
  }}

  function extractPayload(result) {{
    if (result && result.data) {{
      if (result.data["application/json"]) {{
        const v = result.data["application/json"];
        return (typeof v === "string") ? JSON.parse(v) : v;
      }}
      if (result.data["text/plain"]) {{
        let t = result.data["text/plain"];
        t = t.replace(/^['"]/,"").replace(/['"]$/,"");
        t = t.replace(/\\'/g, "'").replace(/\\"/g, '"');
        return JSON.parse(t);
      }}
    }}
    if (result && result.reply) return result;
    throw new Error("Could not parse kernel response: " + JSON.stringify(result).substring(0, 200));
  }}

  window.hfSend = async function() {{
    const text = inputEl.value.trim();
    if (!text) return;
    inputEl.value = "";
    inputEl.disabled = true;
    sendBtn.disabled = true;
    sendBtn.textContent = "...";

    addBubble("user", text);
    history.push({{role: "user", content: text}});
    const thinking = addBubble("assistant", "Thinking...");

    try {{
      const result = await google.colab.kernel.invokeFunction(
        "notebook.chat",
        [JSON.stringify(history)],
        {{}}
      );
      const data = extractPayload(result);
      if (data.error) throw new Error(data.error);
      thinking.textContent = data.reply;
      history.push({{role: "assistant", content: data.reply}});
    }} catch (err) {{
      thinking.textContent = "Error: " + err.message;
      thinking.style.background = "#ffe0e0";
      thinking.style.color = "#900";
      history.pop();
    }}

    inputEl.disabled = false;
    sendBtn.disabled = false;
    sendBtn.textContent = "Send";
    inputEl.focus();
  }};

  inputEl.addEventListener("keydown", (e) => {{
    if (e.key === "Enter" && !e.shiftKey) {{ e.preventDefault(); hfSend(); }}
  }});
}})();
</script>
"""

display(HTML(CHAT_HTML))
print("Chat ready. First response may be slower due to CUDA warm-up.")

---
### Notes

- **T4 (16 GB VRAM) limits** — fp16 fits ~7B models; 4-bit fits ~13B. Check the weight size column before loading.
- **Gated models (🔒)** — Llama, Gemma, Phi-4, etc. require accepting the model license on HuggingFace and logging in via cell 3.
- **OOM errors** — if you get `CUDA out of memory`, either switch to 4-bit or pick a smaller model. Run `torch.cuda.empty_cache()` between attempts.
- **No chat template** — models without a `tokenizer.chat_template` fall back to a plain `User: / Assistant:` format. Quality may degrade for instruction-tuned models that expect a specific template.
- **Quantization note** — 4-bit uses `nf4` (NF4) with double quantization via `bitsandbytes`. This is the same scheme used by QLoRA and works well for inference on instruction-tuned models.